# 📊 Notebook 4: Results & Visualizations
**Project:** A Hybrid Deep Learning Approach for Modelling Global CO₂ Emissions  
**Author:** Hafiza Alishba Naaz | NUST Islamabad 2026

> ⚠️ **Run Notebooks 1, 2 & 3 first** to generate all result CSV files.

## 4.1 Import Libraries & Load Results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load saved results
sarimax_df = pd.read_csv('../results/sarimax_results.csv')
hybrid_df  = pd.read_csv('../results/hybrid_results.csv')

print('SARIMAX Results:')
print(sarimax_df.to_string(index=False))
print('\nHybrid Results:')
print(hybrid_df.to_string(index=False))

## 4.2 Performance Comparison Table

In [ ]:
comparison = pd.merge(
    sarimax_df[['Country', 'MAPE_%', 'RMSE', 'R2']].rename(columns={'MAPE_%':'SARIMAX_MAPE', 'RMSE':'SARIMAX_RMSE', 'R2':'SARIMAX_R2'}),
    hybrid_df[['Country', 'Hybrid_MAPE_%', 'Hybrid_RMSE', 'Hybrid_R2']].rename(columns={'Hybrid_MAPE_%':'Hybrid_MAPE'}),
    on='Country'
)
comparison['RMSE_Reduction_%'] = ((comparison['SARIMAX_RMSE'] - comparison['Hybrid_RMSE']) / comparison['SARIMAX_RMSE'] * 100).round(1)
comparison['MAPE_Delta'] = (comparison['SARIMAX_MAPE'] - comparison['Hybrid_MAPE']).round(2)

print('='*90)
print('SARIMAX vs HYBRID — FULL COMPARISON TABLE')
print('='*90)
print(comparison.to_string(index=False))
print('='*90)
print(f'\nAvg SARIMAX MAPE : {comparison["SARIMAX_MAPE"].mean():.2f}%')
print(f'Avg Hybrid MAPE  : {comparison["Hybrid_MAPE"].mean():.2f}%')
print(f'Avg RMSE Reduction: {comparison["RMSE_Reduction_%"].mean():.1f}%')
print(f'Avg Hybrid R²    : {comparison["Hybrid_R2"].mean():.4f}')

## 4.3 RMSE Comparison Bar Chart (SARIMAX vs Hybrid)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(comparison))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['SARIMAX_RMSE'], width, label='SARIMAX', color='#B5D4F4', edgecolor='white')
bars2 = ax.bar(x + width/2, comparison['Hybrid_RMSE'],  width, label='Hybrid',  color='#1F4E79', edgecolor='white')

ax.set_xlabel('Country'); ax.set_ylabel('RMSE (kt CO₂)')
ax.set_title('RMSE Comparison: SARIMAX vs Hybrid Model', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(comparison['Country'], rotation=30, ha='right')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.annotate(f'{bar.get_height():,.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.annotate(f'{bar.get_height():,.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../results/rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 MAPE Improvement Per Country

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#1D9E75' if v > 0 else '#D85A30' for v in comparison['MAPE_Delta']]
ax.bar(comparison['Country'], comparison['MAPE_Delta'], color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('MAPE Improvement: Hybrid over SARIMAX (positive = better)', fontsize=12)
ax.set_xlabel('Country'); ax.set_ylabel('MAPE Reduction (percentage points)')
ax.set_xticklabels(comparison['Country'], rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/mape_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Future Forecasts (2020–2024)

In [ ]:
# Forecast data from FYP report
future_data = {
    'Country':       ['China', 'United States', 'India', 'Russia', 'Japan',
                      'Iran', 'Indonesia', 'Saudi Arabia', 'South Korea', 'Germany'],
    'Forecast_2024': [10850000, 4720000, 2950000, 1650000, 1050000,
                      780000, 620000, 610000, 600000, 680000],
    'Change_vs_2019': [-0.5, -6.0, 18.0, -1.2, -3.5, 7.5, 5.0, 8.2, -0.8, -2.5]
}

future_df = pd.DataFrame(future_data)
colors_fc = ['#D85A30' if v > 0 else '#1D9E75' for v in future_df['Change_vs_2019']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Forecast values
axes[0].barh(future_df['Country'], future_df['Forecast_2024']/1e6, color='#378ADD', edgecolor='white')
axes[0].set_title('Forecasted CO₂ Emissions 2024 (Million kt)', fontsize=12)
axes[0].set_xlabel('CO₂ Emissions (Million kt)')
for i, v in enumerate(future_df['Forecast_2024']/1e6):
    axes[0].text(v + 0.05, i, f'{v:.2f}M', va='center', fontsize=9)

# Change vs 2019
axes[1].barh(future_df['Country'], future_df['Change_vs_2019'], color=colors_fc, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Change vs 2019 (%)', fontsize=12)
axes[1].set_xlabel('Change (%)')
for i, v in enumerate(future_df['Change_vs_2019']):
    axes[1].text(v + 0.1 if v >= 0 else v - 0.1, i, f'{v:+.1f}%', va='center', fontsize=9, ha='left' if v >= 0 else 'right')

plt.suptitle('CO₂ Emission Forecasts 2020–2024 — Hybrid SARIMAX+LSTM', fontsize=13)
plt.tight_layout()
plt.savefig('../results/future_forecasts_2024.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key insight: Developed economies declining, developing economies growing.')

## 4.6 Final Summary

In [ ]:
print('='*70)
print('FINAL RESULTS SUMMARY — HYBRID SARIMAX + LSTM')
print('='*70)
print(f'Countries analyzed : 10 (top global CO₂ emitters)')
print(f'Train period       : 2000–2014')
print(f'Test period        : 2015–2019')
print(f'Future forecast    : 2020–2024')
print()
print(f'SARIMAX Avg MAPE   : {comparison["SARIMAX_MAPE"].mean():.2f}%')
print(f'Hybrid  Avg MAPE   : {comparison["Hybrid_MAPE"].mean():.2f}%')
print(f'MAPE Improvement   : {comparison["MAPE_Delta"].mean():+.2f} pp')
print(f'Avg RMSE Reduction : {comparison["RMSE_Reduction_%"].mean():.1f}%')
print(f'Avg Hybrid R²      : {comparison["Hybrid_R2"].mean():.4f}')
print()
print('All results saved in results/ folder.')
print('='*70)